# 🚢 Titanic Survival Prediction
### End-to-End Machine Learning | Portfolio Notebook | Final Model V5

**Objective:** Predict passenger survival in Kaggle's [Titanic — Machine Learning from Disaster](https://www.kaggle.com/competitions/titanic) competition using interpretable feature engineering, reproducible validation, and a scikit-learn pipeline.

| Project highlight | Result / method |
|---|---|
| Final algorithm | Random Forest (100 trees, maximum depth 5) |
| Key innovation | Cross-fitted target encoding of family, surname and ticket groups |
| Repeated stratified cross-validation | **84.21% mean accuracy** in the previous V5 experiment |
| Best recorded public Kaggle score | **0.79186** (79.186%) |
| Output | `submission_v5.csv` — 418 predictions |

> **Important:** The CV and Kaggle numbers above record results from the previous experiments. This notebook recomputes CV from the input files. Scores may vary slightly across library versions. The public leaderboard score was obtained from a previous Kaggle submission; running a notebook alone cannot reveal hidden test labels.

**Notebook roadmap:** Data loading → understanding missing values → EDA → feature engineering → leakage-aware preprocessing → V4/V5 comparison → validation diagnostics → final training → Kaggle submission → conclusions.

---
## 1. Reproducible environment

We import data analysis and visualization libraries, plus the tools needed for preprocessing and model evaluation. A fixed random seed makes the train/validation splits and model sampling reproducible.

**Goal:** Have every dependency in one place so the notebook works after **Run All** without relying on variables created in an earlier Kaggle session.

In [ ]:
import os
import base64
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from IPython.display import HTML, display

from sklearn import __version__ as sklearn_version
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    classification_report,
)
from sklearn.model_selection import (
    RepeatedStratifiedKFold,
    StratifiedKFold,
    cross_val_predict,
    cross_val_score,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler, TargetEncoder

RANDOM_STATE = 42
sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.max_columns", 25)

print(f"Pandas: {pd.__version__}")
print(f"Scikit-learn: {sklearn_version}")
print(f"Random state: {RANDOM_STATE}")

**Implementation notes — Environment**

- `pandas` and `numpy` support tabular data analysis; `matplotlib` and `seaborn` provide visualizations.
- `Path` handles filesystem paths, while `display()` produces readable tables in Kaggle notebooks.
- `RANDOM_STATE = 42` makes randomized model fitting and validation splits reproducible. The particular number is arbitrary.
- Library versions are printed to help reproduce results across environments. `TargetEncoder` requires scikit-learn 1.3 or later.

---
## 2. Load and validate the Kaggle data

Kaggle stores competition files under `/kaggle/input/titanic/` when the **Titanic competition dataset** has been attached to the notebook. The loader also searches other input folders to handle a different attachment name.

- **Training data** contains the known target `Survived`.
- **Test data** intentionally has no target column; our job is to predict it.
- We keep `PassengerId` unchanged for the submission.

In [ ]:
def locate_titanic_files():
    """Find train.csv and test.csv in the SAME directory."""
    search_roots = []

    local_override = os.environ.get("TITANIC_DATA_DIR")
    if local_override:
        search_roots.append(Path(local_override))

    search_roots.extend([
        Path("/kaggle/input/titanic"),
        Path("/kaggle/input"),
        Path.cwd(),
    ])

    for root in search_roots:
        if not root.is_dir():
            continue

        pairs = []
        for train_path in root.rglob("train.csv"):
            test_path = train_path.with_name("test.csv")
            if test_path.is_file():
                pairs.append((train_path, test_path))

        if pairs:
            pairs.sort(key=lambda pair: (
                "titanic" not in str(pair[0].parent).lower(),
                len(str(pair[0])),
            ))
            return pairs[0]

    raise FileNotFoundError(
        "Titanic train.csv/test.csv not found. In Kaggle, click Add Input "
        "and attach the Titanic competition dataset, then Run All."
    )


train_path, test_path = locate_titanic_files()
train = pd.read_csv(train_path)
test = pd.read_csv(test_path)

required_inputs = {
    "PassengerId", "Pclass", "Name", "Sex", "Age", "SibSp",
    "Parch", "Ticket", "Fare", "Embarked",
}
for dataset_name, frame in [("train", train), ("test", test)]:
    missing_cols = required_inputs.difference(frame.columns)
    if missing_cols:
        raise ValueError(f"{dataset_name} is missing columns: {sorted(missing_cols)}")

if "Survived" not in train.columns or "Survived" in test.columns:
    raise ValueError("Unexpected target columns. Check the selected train/test CSV files.")

print("Training file:", train_path)
print("Test file:    ", test_path)
print("Training shape:", train.shape)
print("Test shape:    ", test.shape)
print("\nFirst training rows:")
display(train.head(5))

**Implementation notes — Data loading**

- `locate_titanic_files()` searches for `train.csv` and `test.csv` in the same folder, making the notebook less dependent on a specific Kaggle input path.
- `Path.rglob()` searches recursively. An informative `FileNotFoundError` is raised if the competition data has not been attached.
- `pd.read_csv()` loads each file into a DataFrame; the input-column checks detect an incorrect dataset early.
- The training file supplies the `Survived` label, while the test file intentionally does not. The original competition contains 891 training passengers and 418 test passengers.

---
## 3. Data understanding and quality checks

The target is `Survived` (0 = did not survive, 1 = survived). Typical input columns include passenger class (`Pclass`), sex, age, relatives aboard (`SibSp`, `Parch`), ticket fare, cabin and embarkation port.

**Quality questions:** How many passengers survived? Where are values missing? Are there duplicate passenger IDs?

In [ ]:
print("TRAINING DATA TYPES")
train.info()

missing_table = pd.DataFrame({
    "train_missing": train.isna().sum(),
    "test_missing": test.isna().sum(),
})
missing_table["train_missing_pct"] = (
    100 * missing_table["train_missing"] / len(train)
).round(1)
missing_table["test_missing_pct"] = (
    100 * missing_table["test_missing"] / len(test)
).round(1)

print("\nMISSING VALUES BY COLUMN")
display(missing_table.sort_values("train_missing", ascending=False))

print("Duplicate train PassengerIds:", train["PassengerId"].duplicated().sum())
print("Duplicate test PassengerIds: ", test["PassengerId"].duplicated().sum())
print("\nTarget counts:")
print(train["Survived"].value_counts().sort_index())
print(f"Survival rate: {train['Survived'].mean():.2%}")

**Implementation notes — Data quality**

- `DataFrame.info()` summarizes column types and non-null values. `isna().sum()` counts missing entries in each column.
- Missing-value percentages use the number of rows in each dataset separately; training and test sets have different sizes.
- The duplicate checks use `PassengerId`, not passenger names. `value_counts()` shows class balance, and the mean of the binary `Survived` column is the overall survival rate.
- Missing values are **not** imputed globally at this stage. The fitted pipeline learns its imputations from each training fold, helping prevent validation leakage.

---
## 4. Exploratory data analysis (EDA)

We inspect associations between survival and passenger characteristics before fitting the model. These are descriptive patterns, **not proof of causation**.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 9))

# A. Target class balance
survival_counts = train["Survived"].value_counts().reindex([0, 1], fill_value=0)
axes[0, 0].bar(["Did not survive", "Survived"], survival_counts.values,
               color=["#64748b", "#0891b2"])
axes[0, 0].set(title="Survival outcome", ylabel="Number of passengers")
for index, count in enumerate(survival_counts.values):
    axes[0, 0].text(index, count + 5, str(count), ha="center")

# B. Survival by sex
sex_rate = train.groupby("Sex")["Survived"].mean().reindex(["male", "female"])
axes[0, 1].bar(sex_rate.index, 100 * sex_rate.values,
               color=["#64748b", "#0891b2"])
axes[0, 1].set(title="Survival rate by sex", ylabel="Survival rate (%)", ylim=(0, 100))

# C. Survival by passenger class
class_rate = train.groupby("Pclass")["Survived"].mean().reindex([1, 2, 3])
axes[1, 0].bar(["1st", "2nd", "3rd"], 100 * class_rate.values,
               color=["#0891b2", "#60a5fa", "#94a3b8"])
axes[1, 0].set(title="Survival rate by ticket class", ylabel="Survival rate (%)",
               ylim=(0, 100))

# D. Age distribution (known ages only)
for outcome, label, color in [
    (0, "Did not survive", "#64748b"),
    (1, "Survived", "#0891b2"),
]:
    ages = train.loc[train["Survived"] == outcome, "Age"].dropna()
    axes[1, 1].hist(ages, bins=22, density=True, alpha=0.55,
                    label=label, color=color)
axes[1, 1].set(title="Age distribution by outcome", xlabel="Age (years)",
               ylabel="Density")
axes[1, 1].legend()

plt.tight_layout()
plt.show()

**Implementation notes — Exploratory analysis**

- `plt.subplots(2, 2)` creates a four-panel figure covering class balance, sex, passenger class, and age distributions.
- `groupby(...)["Survived"].mean()` calculates observed survival proportions because `Survived` takes values of 0 and 1.
- `reindex()` fixes category order for consistent plots, while multiplication by 100 expresses rates as percentages.
- Missing ages are excluded from the **visualization only**. `density=True` makes the age histograms easier to compare despite unequal class sizes.
- These patterns describe associations; they do not establish causal effects.

---
## 5. Feature engineering

Our final model uses both straightforward passenger characteristics and grouping signals:

| Engineered feature | Definition | Motivation |
|---|---|---|
| `FamilySize` | `SibSp + Parch + 1` | Approximate number of close relatives traveling together, including the passenger |
| `Title` | Honorific extracted from `Name` | Encodes age/social-role information (Mr, Mrs, Miss, Master, Rare) |
| `SurnameKey` | Normalized surname | Detects repeated family names |
| `TicketKey` | Normalized ticket identifier | Detects shared tickets |
| `FamilyTicketKey` | `SurnameKey + '|' + TicketKey` | A more specific family-ticket grouping |

**Only inputs available at prediction time are used.** No label (`Survived`) is referenced inside the feature engineering function.

In [ ]:
def engineer_features(df):
    """Create the exact feature set used in the final V5 experiment."""
    df = df.copy()

    df["FamilySize"] = df["SibSp"] + df["Parch"] + 1

    title = df["Name"].str.extract(r",\s*([^.]*)\.")[0].str.strip()
    title = title.replace({"Mlle": "Miss", "Ms": "Miss", "Mme": "Mrs"})
    df["Title"] = title.where(title.isin(["Mr", "Mrs", "Miss", "Master"]), "Rare")

    df["SurnameKey"] = (
        df["Name"].str.split(",", n=1).str[0].str.strip().str.lower()
    )
    df["TicketKey"] = df["Ticket"].astype(str).str.strip().str.upper()
    df["FamilyTicketKey"] = df["SurnameKey"] + "|" + df["TicketKey"]

    return df


train_fe = engineer_features(train)
test_fe = engineer_features(test)

print("Feature-engineered training data:", train_fe.shape)
print("Feature-engineered test data:    ", test_fe.shape)

display(train_fe[[
    "Name", "Title", "SibSp", "Parch", "FamilySize",
    "Ticket", "SurnameKey", "TicketKey", "FamilyTicketKey"
]].head(5))

**Implementation notes — Feature engineering**

- `engineer_features()` applies identical transformations to the training and test datasets. `df.copy()` avoids modifying the original DataFrame.
- `FamilySize` is calculated as `SibSp + Parch + 1`, including the passenger. It is an approximation of close relatives aboard, not necessarily the number sharing one ticket.
- A regular expression extracts honorifics from `Name`. Equivalent titles are normalized, and uncommon titles become `Rare`.
- Splitting `Name` at its first comma extracts a normalized surname. Ticket identifiers are trimmed and uppercased for consistent matching.
- `FamilyTicketKey` combines surname and ticket. All engineered inputs are available at prediction time; **none directly uses the survival label**.

### 5.1 Inspect engineered patterns and train–test group overlap

Many Kaggle test passengers share group identifiers with training passengers. This helps motivate our later use of a group encoder, but it can also mean that ordinary random CV is optimistic for passengers from entirely new groups.

In [ ]:
title_summary = (
    train_fe.groupby("Title")["Survived"]
    .agg(passengers="size", survival_rate="mean")
    .sort_values("survival_rate", ascending=False)
)
print("SURVIVAL BY PASSENGER TITLE")
display(title_summary.round(3))

family_summary = (
    train_fe.groupby("FamilySize")["Survived"]
    .agg(passengers="size", survival_rate="mean")
    .sort_index()
)

fig, ax = plt.subplots(figsize=(9, 4))
ax.bar(family_summary.index.astype(str), 100 * family_summary["survival_rate"],
       color="#0891b2")
ax.set(title="Survival rate by approximate family size", xlabel="FamilySize",
       ylabel="Survival rate (%)", ylim=(0, 100))
plt.tight_layout()
plt.show()

same_family_ticket = test_fe["FamilyTicketKey"].isin(train_fe["FamilyTicketKey"])
same_surname = test_fe["SurnameKey"].isin(train_fe["SurnameKey"])
same_ticket = test_fe["TicketKey"].isin(train_fe["TicketKey"])

overlap = pd.DataFrame({
    "Matching train group": [
        "Exact surname + ticket", "Surname", "Ticket", "Surname OR ticket"
    ],
    "Test passengers": [
        same_family_ticket.sum(), same_surname.sum(), same_ticket.sum(),
        (same_surname | same_ticket).sum()
    ],
})
overlap["Percent of test"] = (
    100 * overlap["Test passengers"] / len(test_fe)
).round(1)
print("TRAIN / TEST GROUP OVERLAP")
display(overlap)

**Implementation notes — Group patterns**

- `groupby(...).agg()` reports both sample size and observed survival rate. Rates for very small groups should be interpreted cautiously.
- The family-size chart explores descriptive associations, while `.isin()` measures whether test-set group identifiers also appear in the training set.
- The union of matching surnames and matching tickets is calculated with the logical OR operator (`|`). This overlap analysis does not consult test labels.
- Group survival statistics are **not manually mapped from the full training set** into validation features. Instead, a cross-fitted `TargetEncoder` is used within the model pipeline to reduce target leakage.

---
## 6. Select features and build a leakage-aware pipeline

We freeze the chosen V5 configuration, without introducing the discarded `FarePerFamilyMember` or smart-age imputation features.

**Final feature inventory**

- Numerical: `Pclass`, `Age`, `Fare`, `FamilySize`
- Categorical: `Sex`, `Embarked`, `Title`
- Group identifiers: `FamilyTicketKey`, `SurnameKey`, `TicketKey`

The `Pipeline` learns missing-value imputation, categorical encoding, and group target encoding **inside every training fold** before fitting the Random Forest.

In [ ]:
NUMERIC_FEATURES = ["Pclass", "Age", "Fare", "FamilySize"]
CATEGORICAL_FEATURES = ["Sex", "Embarked", "Title"]
GROUP_FEATURES_V4 = ["FamilyTicketKey"]
GROUP_FEATURES_V5 = ["FamilyTicketKey", "SurnameKey", "TicketKey"]

BASE_FEATURES = NUMERIC_FEATURES + CATEGORICAL_FEATURES
V4_FEATURES = BASE_FEATURES + GROUP_FEATURES_V4
V5_FEATURES = BASE_FEATURES + GROUP_FEATURES_V5

X_base = train_fe[BASE_FEATURES].copy()
X_v4 = train_fe[V4_FEATURES].copy()
X_v5 = train_fe[V5_FEATURES].copy()
X_test_v5 = test_fe[V5_FEATURES].copy()
y = train_fe["Survived"].astype(int).copy()


def build_rf_pipeline(group_columns=None):
    """Return a fresh Random Forest pipeline with optional group target encoding."""
    group_columns = list(group_columns) if group_columns else []

    numeric_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ])

    categorical_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OneHotEncoder(handle_unknown="ignore")),
    ])

    transformers = [
        ("numeric", numeric_pipeline, NUMERIC_FEATURES),
        ("categorical", categorical_pipeline, CATEGORICAL_FEATURES),
    ]

    if group_columns:
        transformers.append((
            "group_survival",
            TargetEncoder(target_type="binary", cv=5, random_state=RANDOM_STATE),
            group_columns,
        ))

    return Pipeline([
        ("preprocessor", ColumnTransformer(transformers, remainder="drop")),
        ("model", RandomForestClassifier(
            n_estimators=100,
            max_depth=5,
            random_state=RANDOM_STATE,
        )),
    ])


print("Base training features:", X_base.shape)
print("V4 training features:  ", X_v4.shape)
print("V5 training features:  ", X_v5.shape)
print("V5 test features:      ", X_test_v5.shape)
print("Target distribution:   ", y.value_counts().to_dict())

**Implementation notes — Leakage-aware modeling pipeline**

1. **Feature separation:** `X_base`, `X_v4`, and `X_v5` contain only predictive inputs; `y` contains the training labels. `X_test_v5` uses the same feature columns as the V5 training input.
2. **Numeric processing:** `SimpleImputer(strategy="median")` replaces missing numeric values using medians learned from the current training fold. `StandardScaler` standardizes numeric values; scaling is not essential for Random Forests but is retained to match the original V5 experiment.
3. **Categorical processing:** Missing categorical entries use the most frequent value. `OneHotEncoder(handle_unknown="ignore")` converts categories into numeric indicators without failing on categories absent during fitting.
4. **Group processing:** `TargetEncoder(target_type="binary", cv=5)` encodes the family, surname, and ticket identifiers using their relationship with survival. Its internal cross-fitting during `fit_transform()` reduces direct self-target leakage.
5. **Combined preprocessing:** `ColumnTransformer` applies each transformation to the relevant columns. The resulting features feed a `RandomForestClassifier` with **100 trees** and **maximum depth 5**.
6. **Comparable experiments:** `build_rf_pipeline()` returns an unfitted pipeline so the baseline, V4, and V5 models can be evaluated with identical forest settings.

**Validation safeguard:** Cross-validation fits preprocessing on the training portion of each outer fold and transforms its validation portion separately. Cross-fitting does not, however, make passengers with shared family/ticket identifiers statistically independent. Random-fold performance should not be interpreted as a guarantee for completely unseen groups.

---
## 7. Fair comparison: baseline vs V4 vs V5

We evaluate all three pipelines on **the same 15 validation splits**: 5 stratified folds × 3 repeats, shuffled with random state 42. Accuracy is Kaggle Titanic's main competition metric.

Comparison logic:

1. Baseline: no group target encoding.
2. V4: only the combined family-ticket key.
3. **V5: all three keys** (our chosen final model).

This cell may take a little time because it fits 45 Random Forest pipelines.

In [ ]:
cv = RepeatedStratifiedKFold(
    n_splits=5,
    n_repeats=3,
    random_state=RANDOM_STATE,
)

experiments = {
    "Baseline (no group keys)": (build_rf_pipeline(), X_base),
    "V4 (family + ticket key)": (build_rf_pipeline(GROUP_FEATURES_V4), X_v4),
    "V5 (expanded group keys)": (build_rf_pipeline(GROUP_FEATURES_V5), X_v5),
}

cv_scores = {}
for experiment_name, (candidate_model, candidate_X) in experiments.items():
    scores = cross_val_score(
        candidate_model,
        candidate_X,
        y,
        cv=cv,
        scoring="accuracy",
        n_jobs=1,
    )
    cv_scores[experiment_name] = scores
    print(f"{experiment_name}: {scores.mean():.2%} ± {scores.std():.2%}")

cv_summary = pd.DataFrame([
    {
        "Experiment": name,
        "CV mean accuracy (%)": round(100 * scores.mean(), 2),
        "CV standard deviation (pp)": round(100 * scores.std(), 2),
        "Number of folds": len(scores),
    }
    for name, scores in cv_scores.items()
]).sort_values("CV mean accuracy (%)", ascending=False)

print("\nCROSS-VALIDATION COMPARISON")
display(cv_summary)

**Implementation notes — Cross-validation**

- `RepeatedStratifiedKFold(n_splits=5, n_repeats=3)` produces **15 validation scores** per model, preserving approximately the same target-class proportions in every fold.
- A dictionary stores the three experiments, and a loop evaluates each one through `cross_val_score(..., scoring="accuracy")` on identical splits.
- The **mean** summarizes average fold accuracy; the **standard deviation** indicates fold-to-fold variation. Neither replaces an independent final evaluation.
- `n_jobs=1` favors predictable execution across environments.
- Earlier experiments produced approximately **82.98%** for the baseline, **83.80%** for V4, and **84.21%** for V5. The executed cell, rather than these historical figures, is the source of the current run's results.

### 7.1 What improved, fold by fold?

A small rise in average accuracy does not always mean consistent improvement. We compare V5 and V4 on paired splits and examine the distribution of fold scores.

In [ ]:
v4_fold_scores = cv_scores["V4 (family + ticket key)"]
v5_fold_scores = cv_scores["V5 (expanded group keys)"]
fold_differences = 100 * (v5_fold_scores - v4_fold_scores)

better = int((fold_differences > 0).sum())
tied = int((fold_differences == 0).sum())
worse = int((fold_differences < 0).sum())

print(f"V5 better than V4: {better} / {len(fold_differences)} folds")
print(f"Equal: {tied} folds; worse: {worse} folds")
print(f"Average difference: {fold_differences.mean():+.2f} percentage points")

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].boxplot(
    [100 * cv_scores[name] for name in cv_scores],
    showmeans=True,
)
axes[0].set_xticks([1, 2, 3], labels=["Baseline", "V4", "V5"])
axes[0].set(title="Accuracy across 15 CV folds", ylabel="Accuracy (%)")

axes[1].axhline(0, color="#64748b", linewidth=1)
axes[1].bar(
    range(1, len(fold_differences) + 1),
    fold_differences,
    color=["#0891b2" if diff >= 0 else "#dc2626" for diff in fold_differences],
)
axes[1].set(title="Paired V5 minus V4 (same splits)",
            xlabel="Fold number", ylabel="Accuracy difference (percentage points)")
plt.tight_layout()
plt.show()

**Implementation notes — Paired model comparison**

- Subtracting V4's score from V5's score on the **same fold** yields a paired difference in percentage points.
- Counts of positive, zero, and negative differences show how consistently the expanded group features help, rather than relying on the average alone.
- The boxplots summarize fold-score distributions; the second chart visualizes each paired difference.
- In the earlier experiment, V5 improved on **9 of 15 folds**, with a mean advantage of **0.41 percentage points**. This is encouraging but modest.
- Because multiple features were evaluated during development, model-selection bias remains a limitation.

---
## 8. Out-of-fold error analysis

For a simple diagnostic, we make **one** 5-fold set of out-of-fold (OOF) predictions and build a confusion matrix. Each training passenger is predicted by a model fitted without that passenger's validation fold.

This is **not** the same statistic as the 15-fold repeated mean shown above.

In [ ]:
diagnostic_cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=RANDOM_STATE,
)

oof_predictions = cross_val_predict(
    build_rf_pipeline(GROUP_FEATURES_V5),
    X_v5,
    y,
    cv=diagnostic_cv,
    method="predict",
    n_jobs=1,
)

print(f"OOF accuracy (one 5-fold split): {accuracy_score(y, oof_predictions):.2%}")
print("\nCLASSIFICATION REPORT")
print(classification_report(
    y,
    oof_predictions,
    target_names=["Did not survive", "Survived"],
    digits=3,
))

fig, ax = plt.subplots(figsize=(6, 5))
ConfusionMatrixDisplay.from_predictions(
    y,
    oof_predictions,
    display_labels=["Did not survive", "Survived"],
    cmap="Blues",
    colorbar=False,
    ax=ax,
)
ax.set_title("V5 — Out-of-fold confusion matrix")
plt.tight_layout()
plt.show()

**Implementation notes — Error analysis**

- A separate `StratifiedKFold` with five folds is used for **out-of-fold (OOF)** predictions, ensuring each passenger is predicted by a model that did not train on that passenger's outer validation fold.
- `classification_report()` adds **precision**, **recall**, and **F1-score** to the headline accuracy measure.
- The confusion matrix shows true negatives, false positives, false negatives, and true positives.
- This diagnostic 5-fold OOF accuracy need not equal the earlier **15-fold repeated-CV mean** because the evaluation procedures differ. The analysis is used to understand errors, not to tune the final chosen model again.

---
## 9. Fit the final V5 model on all available labeled data

Once evaluation is complete, we fit a **new** V5 pipeline on all 891 labeled rows so it can predict the separate, unlabeled Kaggle test data.

We deliberately preserve the exact V5 hyperparameters that produced our best public submission.

In [ ]:
final_model_v5 = build_rf_pipeline(GROUP_FEATURES_V5)
final_model_v5.fit(X_v5, y)

final_test_predictions = final_model_v5.predict(X_test_v5).astype(int)

assert len(final_test_predictions) == len(test_fe)
assert np.isin(final_test_predictions, [0, 1]).all()

print("Model fitted successfully.")
print("Predictions generated:", len(final_test_predictions))
print("Prediction distribution:")
print(pd.Series(final_test_predictions, name="Survived").value_counts().sort_index())

**Implementation notes — Final fitting and prediction**

- Calling `build_rf_pipeline(GROUP_FEATURES_V5)` creates a **fresh model**, distinct from the fitted models used in cross-validation.
- `fit(X_v5, y)` learns preprocessing parameters and trains the forest on **all available labeled training passengers**.
- `predict(X_test_v5)` produces a binary prediction for every Kaggle test passenger; `astype(int)` ensures integer submission labels.
- Assertions check the number and allowed values of predictions. Their distribution is a sanity check, **not an accuracy estimate**, since the true test labels are unavailable.

### 9.1 What did the forest use? (descriptive feature importance)

Tree-based feature importances can help inspect the fitted model. However, impurity-based importance can favor some feature structures, and **importance is not causation**. Interpret these results as model diagnostics only.

In [ ]:
fitted_preprocessor = final_model_v5.named_steps["preprocessor"]
fitted_forest = final_model_v5.named_steps["model"]

transformed_feature_names = fitted_preprocessor.get_feature_names_out()
feature_importance = pd.Series(
    fitted_forest.feature_importances_,
    index=transformed_feature_names,
).sort_values(ascending=False)

print("TOP 12 MODEL FEATURES")
display(feature_importance.head(12).rename("importance").to_frame())

fig, ax = plt.subplots(figsize=(10, 6))
feature_importance.head(12).sort_values().plot.barh(
    ax=ax, color="#0891b2"
)
ax.set(title="V5 — Random Forest feature importance (top 12)",
       xlabel="Impurity-based importance", ylabel="Encoded feature")
plt.tight_layout()
plt.show()

**Implementation notes — Model interpretation**

- `named_steps` accesses the fitted preprocessor and forest inside the pipeline.
- `get_feature_names_out()` retrieves feature names **after** imputation and encoding, including one-hot and target-encoded columns.
- `feature_importances_` contains impurity-based importance scores from the forest. A `Series` pairs the scores with the transformed feature names and sorts them for inspection.
- Importance rankings are descriptive model diagnostics. Correlated predictors and feature structure can affect them; they must **not** be interpreted as causal explanations.

---
## 10. Create and validate the Kaggle submission

Kaggle expects a CSV with **exactly two columns**:

1. `PassengerId` — unchanged passenger IDs from `test.csv`.
2. `Survived` — binary 0/1 predictions.

We save the result in `/kaggle/working/` so it appears in the notebook's **Output** panel after execution.

In [ ]:
submission_v5 = pd.DataFrame({
    "PassengerId": test_fe["PassengerId"].to_numpy(),
    "Survived": final_test_predictions,
})

assert list(submission_v5.columns) == ["PassengerId", "Survived"]
assert len(submission_v5) == len(test_fe)
assert submission_v5["PassengerId"].equals(test_fe["PassengerId"].reset_index(drop=True))
assert submission_v5.notna().sum().sum() == submission_v5.size
assert submission_v5["Survived"].isin([0, 1]).all()

output_dir = Path("/kaggle/working") if Path("/kaggle/working").is_dir() else Path.cwd()
submission_path = output_dir / "submission_v5.csv"
submission_v5.to_csv(submission_path, index=False)

print("Saved Kaggle submission:", submission_path)
print("Submission shape:", submission_v5.shape)
print("Missing cells:", submission_v5.isna().sum().sum())
display(submission_v5.head(10))

**Implementation notes — Submission format**

- `pd.DataFrame()` constructs the **two required Kaggle columns**: `PassengerId` and `Survived`.
- `to_numpy()` preserves test passenger IDs in their original order; these IDs must not be replaced with DataFrame index values.
- Assertions verify column names, row count, ID alignment, completeness, and binary predictions.
- `to_csv(index=False)` omits the unnecessary pandas index, avoiding an invalid third CSV column.
- On Kaggle, the output is written to `/kaggle/working/submission_v5.csv`; the expected competition shape is **(418, 2)**.

### 10.1 Direct CSV download link (optional)

For notebook interfaces where downloading files from the Output panel is inconvenient, this cell displays a small direct-download link as an alternative. The CSV is already saved even if the link is blocked by the browser.

In [ ]:
encoded_submission_csv = base64.b64encode(
    submission_path.read_bytes()
).decode("ascii")

display(HTML(
    '<a href="data:text/csv;base64,' + encoded_submission_csv + '" '
    'download="submission_v5.csv">'
    '⬇ Download submission_v5.csv'
    '</a>'
))

print("If the browser blocks the link, download submission_v5.csv from Kaggle Output.")

**Implementation notes — Optional direct download**

- `submission_path.read_bytes()` reads the generated CSV, and `base64.b64encode()` converts its bytes into a text payload usable in an HTML download link.
- `HTML()` and `display()` show the link directly below the cell. Its filename is specified with the `download` attribute.
- The variable `encoded_submission_csv` deliberately has a descriptive name to avoid overwriting the training DataFrame.
- This convenience step is **optional**: the CSV was already saved and can also be downloaded from Kaggle's Output panel.

---
## 11. Experiment history and final conclusions

### Previous Kaggle submission history

The scores below come from the earlier Kaggle submissions for this project. **They are historical results, not scores produced by the current execution.**

| Version | Main approach | Public Kaggle accuracy |
|---|---|---:|
| V1 | Logistic Regression baseline | 0.76555 |
| V2 | Random Forest with baseline engineered features | 0.77511 |
| V3 | Tuned Gradient Boosting | 0.77272 |
| V4 | Random Forest + `FamilyTicketKey` target encoding | 0.78468 |
| **V5** | **Random Forest + three group keys** | **0.79186** |

### Other feature and modeling experiments

| Experiment | Earlier CV result | Decision |
|---|---|---|
| V4 group-encoded baseline | 83.80% | Replaced by expanded V5 |
| Smart age imputation (Title + Pclass), compared to V4 | 83.87% (+0.07 pp; 4/15 folds improved) | Not adopted |
| **V5 expanded groups** | **84.21%** | **Selected** |
| Fare per family member added to V5 | 83.95% (−0.26 pp; 2/15 folds improved) | Rejected |
| Soft-voting ensemble with Random Forest + XGBoost | 84.10% (−0.11 pp; 4/15 folds improved) | Rejected |

### What this project demonstrates

- **Feature engineering can matter more than algorithm switching.** The improvement from V4 to V5 came from more informative grouping features while retaining the same Random Forest hyperparameters.
- **Reliable preprocessing belongs inside validation.** Median/mode imputation, one-hot encoding and target encoding are all fitted within the pipeline.
- **Cross-validation and leaderboard scores answer different questions.** Repeated random CV estimates performance on holdouts from the labeled dataset; the public Kaggle score uses a hidden subset of `test.csv`.
- **Negative experiments are valuable.** Smart age imputation, fare-per-family-member, and the tested ensemble did not justify replacing V5.

### Limitations and next steps

- Titanic is a small dataset. Small differences such as **0.41 percentage points** can reflect noise or model-selection effects.
- Random stratified folds can contain passengers from overlapping family/ticket groups; performance on **new, unseen groups** may be lower.
- The public Kaggle score is **not** the full hidden-test accuracy, and none of these associations prove causation.
- Further development would ideally use an additional untouched evaluation protocol, but **the agreed final portfolio model is V5** rather than another round of leaderboard-driven tuning.

**Final deliverable:** a reproducible Kaggle notebook and a validated `submission_v5.csv` generated by its final cells.
